<a href="https://colab.research.google.com/github/CBravoR/AdvancedAnalyticsLabs/blob/master/notebooks/python/Lab_Model_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Validation

In this lab we validate the models we built in the previous labs. We follow the structure of the lecture: three **levels** of backtesting (Level 0, data stability; Level 1, discrimination; Level 2, calibration), then LGD backtesting, then benchmarking, and we close with the traffic-light dashboard and the action scheme.

Two things make validation different from model building:

1. It is **out-of-time**. We fix everything (grades, long-run PDs, thresholds) on a development window and then look at what happened afterwards, one year at a time.
2. It produces **decisions**, not metrics. Every test ends in a colour, and every colour is tied to an action.

We use three datasets, all from earlier labs:

- `PDCalExample.xlsx`, the monthly panel from the PD calibration lab (108 months, about 80 loans a month, 557 defaults). It gives us scores, defaults and time, so it carries Levels 1 and 2 and the score-level part of Level 0.
- The **bankloan** data from Labs 6 and 7 (a scorecard and an XGBoost model). It has the input variables, so it carries the attribute-level part of Level 0, the SHAP check of Level 1 and the benchmarking section.
- `LGD.csv` from the LGD lab, for the LGD backtests.

The PD panel is small. With roughly 80 loans and 5 defaults a month, no monthly test has power, so we aggregate to **years**. That is the "data aggregation" slide of the lecture, in code.

In [ ]:
# Data files from the previous labs, in this order:
# PDCalExample.xlsx (PD calibration lab), LGD.csv (LGD lab),
# train_woe.parquet, test_woe.parquet and BankloanCleanNewVars.pkl (Labs 5 to 7)
!gdown 'https://drive.google.com/uc?id=1UYmgsu5gI5U_VbraKXHxWTXyZSbM6q5S'
!gdown 'https://drive.google.com/uc?id=1nldxUFNGDziLZgE-fJv5KmNjnbdM29na'
!gdown 'https://drive.google.com/uc?id=12AFRYPBY6N_hnvZJkSDL_nhWwjt43M-n'
!gdown 'https://drive.google.com/uc?id=1IEvsKnMMwHrOsqR1EaaaMQcms1vTiQgu'
!gdown 'https://drive.google.com/uc?id=1aDraDSR2OQbIMjIY07s-rD5cel2x_iS-'

In [ ]:
# Install latest version of SHAP library
%pip install -q shap

In [ ]:
# Data wrangling
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import pandas as pd
import polars as pl

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

# Statistics
from scipy import stats
from scipy.spatial.distance import jensenshannon

# Models and metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, ElasticNetCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier, XGBRegressor

# Colours for the traffic lights, used throughout
LIGHTS = {'green': '#8fd18f', 'yellow': '#f7e463', 'orange': '#f5a94a', 'red': '#f28b82', 'grey': '#dddddd'}

def paint(df, col='light'):
    """Colour a dataframe by the value in column `col`."""
    def _row(r):
        c = LIGHTS.get(r[col], 'white')
        return [f'background-color: {c}' if k == col else '' for k in r.index]
    return df.style.apply(_row, axis=1).format(precision=4)

rng = np.random.default_rng(20260909)

## 0. Setup: a development window and a monitoring window

The PD panel covers 108 months, from January 1999 (portfolio 1). We treat the first four years (months 1 to 48) as the **development sample**: it is where the rating grades and their long-run PDs are set. The remaining five years (months 49 to 108) are the **monitoring sample**. Every test below asks the same question: given what we fixed at development, does year *t* look like what we promised?

In [ ]:
# Load the PD calibration data
loans = pd.read_excel('PDCalExample.xlsx', sheet_name=0)
econ  = pd.read_excel('PDCalExample.xlsx', sheet_name=1)

# Portfolio 1 is January 1999; a "year" is 12 consecutive portfolios.
loans['Year'] = 1999 + (loans['Portfolio'] - 1) // 12
DEV_YEARS = [1999, 2000, 2001, 2002]
MON_YEARS = sorted(loans.loc[~loans['Year'].isin(DEV_YEARS), 'Year'].unique())

# Split the data into development and monitoring portfolios
dev = loans[loans['Year'].isin(DEV_YEARS)].copy()
mon = loans[~loans['Year'].isin(DEV_YEARS)].copy()

# Summarise the data by year
summary = loans.groupby('Year').agg(loans=('Default', 'size'), defaults=('Default', 'sum'), DR=('Default', 'mean'))
summary['window'] = np.where(summary.index.isin(DEV_YEARS), 'development', 'monitoring')
summary

### Rebuilding the rating grades

The PD calibration lab found the grade boundaries by segmenting the ROC curve with `pwlf`. That fit is slow, so we reuse its breakpoints (the `res` vector saved in that lab) and map them back to score thresholds on the **development** ROC curve. Grades are named G1 (safest) to GK (riskiest). If a grade has no defaults in development, we merge it with its riskier neighbour, as we did in the calibration lab.

In [ ]:
# Define the score cuts for the grades.
res = np.array([0., 0.01383389, 0.03920524, 0.07731607, 0.11285577,
                0.20653665, 0.32339978, 0.41818781, 0.57283343, 0.7999917, 1.])

# Compute the thresholds corresponding to the desired PDs on the ROC curve.
fpr, tpr, thr = roc_curve(dev['Default'], dev['Probs'])
cuts = np.array([thr[np.abs(fpr - r).argmin()] for r in res])
cuts = np.flip(cuts)               # thresholds decrease along the ROC curve; we want increasing PD
cuts = np.insert(cuts, 0, 0.0)
cuts[-1] = 1.0

# Assign grades to the loans based on the score cuts
def assign_grades(probs, cuts):
    labels = [f'G{i+1}' for i in range(len(cuts) - 1)]
    return pd.cut(probs, cuts, labels=labels, include_lowest=True).astype(str)

# Merge grades without defaults in development into the next riskier grade
while True:
    g = assign_grades(dev['Probs'], cuts)
    labels = [f'G{i+1}' for i in range(len(cuts) - 1)]
    d_by_grade = dev.groupby(g)['Default'].sum().reindex(labels).fillna(0)
    empty = [k for k in labels if d_by_grade[k] == 0]
    if not empty:
        break
    idx = int(empty[0][1:])       # grade number, 1-based; its upper cut is cuts[idx]
    cuts = np.delete(cuts, idx)

# Assign grades to the loans based on the score cuts
GRADES = [f'G{i+1}' for i in range(len(cuts) - 1)]
loans['Grade'] = assign_grades(loans['Probs'], cuts)
dev['Grade']   = assign_grades(dev['Probs'], cuts)
mon['Grade']   = assign_grades(mon['Probs'], cuts)

# Summarise the data by grade
print(f'{len(GRADES)} grades. Score cuts:')
print(np.round(cuts, 4))

The **long-run PD** of each grade is the default-count weighted default rate over the development window (the same estimator as in the calibration lab). This is the number the model "promises", and the number every Level 2 test is measured against.

In [ ]:
# Summarise the development data by grade
master = dev.groupby('Grade').agg(n=('Default', 'size'), d=('Default', 'sum')).reindex(GRADES)
master['PD_TTC'] = master['d'] / master['n']
master['share']  = master['n'] / master['n'].sum()
master.style.format({'PD_TTC': '{:.4f}', 'share': '{:.3f}'})

Note the small numbers. G1 to G3 hold most of the portfolio and a handful of defaults each. **Any per-grade test will have little power there**; that fact will shape how we read the dashboard later.

## 1. Level 0: data stability

Level 0 asks whether the population the model sees now is the population it was built on. The lecture uses the **system stability index** (SSI, also called PSI) with the 0.10 / 0.25 rule of thumb. We compute it on the score distribution (deciles fixed on the development sample) and on the grade distribution, for each monitoring year.

$$\text{SSI} = \sum_j (A_j - T_j)\,\ln\frac{A_j}{T_j}$$

where $T_j$ is the development share of bin $j$ and $A_j$ the current share.

In [ ]:
# Define functions for the dashboard
def shares(x, bins=None, labels=None):
    """Share of observations per bin. `bins` are cut points; if None, `x` is already categorical."""
    if bins is None:
        s = pd.Series(x).value_counts(normalize=True)
    else:
        s = pd.cut(x, bins, include_lowest=True).value_counts(normalize=True)
    if labels is not None:
        s = s.reindex(labels).fillna(0.0)
    return s.sort_index()

def ssi(actual, expected, eps=1e-6):
    a = np.clip(np.asarray(actual, float), eps, None)
    t = np.clip(np.asarray(expected, float), eps, None)
    return float(np.sum((a - t) * np.log(a / t)))

def ssi_light(v):
    return 'green' if v < 0.10 else ('yellow' if v < 0.25 else 'red')

# Score deciles fixed on the development sample
dec_bins = np.quantile(dev['Probs'], np.linspace(0, 1, 11))
dec_bins[0], dec_bins[-1] = 0.0, 1.0
T_score = shares(dev['Probs'], dec_bins)
T_grade = shares(dev['Grade'], labels=GRADES)

rows = []
for y in MON_YEARS:
    cur = mon[mon['Year'] == y]
    A_score = shares(cur['Probs'], dec_bins)
    A_grade = shares(cur['Grade'], labels=GRADES)
    rows.append({'Year': y,
                 'SSI score deciles': ssi(A_score, T_score),
                 'SSI grades': ssi(A_grade, T_grade)})
ssi_tab = pd.DataFrame(rows).set_index('Year')
ssi_tab['light'] = ssi_tab['SSI grades'].map(ssi_light)
paint(ssi_tab)

### Limitations of the SSI

The SSI has no reference distribution, it is unbounded, and it explodes when a bin is nearly empty. The lecture lists three alternatives with better-known behaviour:

- the **Jensen–Shannon divergence**, symmetric and bounded in $[0, \ln(2)]$;
- the **Kolmogorov–Smirnov** statistic and the **Wasserstein distance** on the continuous score, which need no binning at all;
- **adversarial validation**, which we run in the bankloan annex below.

Let's see how they move together on our panel.

In [ ]:
rows = []
for y in MON_YEARS:
    cur = mon[mon['Year'] == y]
    A_score = shares(cur['Probs'], dec_bins)
    ks = stats.ks_2samp(cur['Probs'], dev['Probs'])
    rows.append({'Year': y,
                 'SSI (deciles)': ssi(A_score, T_score),
                 'JSD (deciles)': jensenshannon(A_score, T_score, base=np.e) ** 2,   # squared distance = divergence
                 'KS statistic': ks.statistic, 'KS p-value': ks.pvalue,
                 'Wasserstein': stats.wasserstein_distance(cur['Probs'], dev['Probs'])})
pd.DataFrame(rows).set_index('Year').style.format(precision=4)

The KS test is the only one of the four that comes with a p-value out of the box. On this panel the shifts are small, and all four measures agree. They disagree when bins are thin: the SSI grows without bound while the JSD saturates. Try recomputing the SSI with 20 bins instead of 10 to see it.

### Concentration in rating grades

A rating system that piles obligors into a few grades stops differentiating risk, even if every test above passes. The ECB's validation reporting uses the **Herfindahl index** of the grade shares $R_k$, normalised to $[0, 1]$:

$$\text{HI} = 1 + \frac{\ln\!\left(\frac{CV^2 + 1}{K}\right)}{\ln K}, \qquad CV = \frac{\sqrt{\frac{1}{K}\sum_k (R_k - \frac{1}{K})^2}}{1/K}$$

We compare the index of each monitoring year with the development value. The ECB sets no threshold; we bootstrap a 95% interval for the development value and flag years that fall outside it.

In [ ]:
# Define the Herfindahl index
def herfindahl(shares_):
    R = np.asarray(shares_, float); K = len(R)
    cv = np.sqrt(np.mean((R - 1 / K) ** 2)) / (1 / K)
    return 1 + np.log((cv ** 2 + 1) / K) / np.log(K)

# Compute the Herfindahl index for the development sample and a bootstrap 95% confidence interval
HI_dev = herfindahl(T_grade)
boot = np.array([herfindahl(shares(dev['Grade'].sample(len(dev), replace=True, random_state=s), labels=GRADES)) for s in range(500)])
lo, hi = np.quantile(boot, [0.025, 0.975])

rows = []
for y in MON_YEARS:
    cur = mon[mon['Year'] == y]
    h = herfindahl(shares(cur['Grade'], labels=GRADES))
    rows.append({'Year': y, 'HI': h, 'HI development': HI_dev,
                 'light': 'green' if lo <= h <= hi else ('yellow' if abs(h - HI_dev) < 0.10 else 'red')})
print(f'Development HI = {HI_dev:.3f}, bootstrap 95% interval [{lo:.3f}, {hi:.3f}]')
paint(pd.DataFrame(rows).set_index('Year'))

The index rises in every monitoring year: the portfolio moved towards the safer grades, so the grade distribution is more concentrated than at development. The bootstrap interval only captures sampling noise, so any structural change flags. Whether that is a problem is a judgement call, which is why the ECB reports the number without a threshold. The decision for monitoring and alerts is left to the bank.

### Annex A: attribute-level stability on the bankloan data

The PD panel has no input variables, so for the attribute-level tests (Step 2 of Level 0 in the lecture) we use the bankloan scorecard data from Lab 6. The WoE-transformed training set plays the role of the development sample and the test set plays the role of the current sample. Each WoE value is a bin, so the SSI per attribute can be directly calculated from those bins. We add the **t test** on each attribute's mean, which is the third quantitative row of the Level 0 dashboard.

In [ ]:
bl_train = pl.read_parquet('train_woe.parquet').to_pandas()
bl_test  = pl.read_parquet('test_woe.parquet').to_pandas()
woe_cols = [c for c in bl_train.columns if c != 'Default']

rows = []
for c in woe_cols:
    labels = sorted(set(bl_train[c].round(6)) | set(bl_test[c].round(6)))
    T = shares(bl_train[c].round(6), labels=labels)
    A = shares(bl_test[c].round(6), labels=labels)
    t = stats.ttest_ind(bl_test[c], bl_train[c], equal_var=False)
    rows.append({'attribute': c, 'bins': len(labels), 'SSI': ssi(A, T),
                 't p-value': t.pvalue,
                 'light': ssi_light(ssi(A, T))})
paint(pd.DataFrame(rows).set_index('attribute'))

Everything is green. That is expected: the test set is a random split of the same population, so there is no shift to find. That makes it a good place to see what a **shift** looks like. Below we build a drifted "current" sample by over-sampling young borrowers with high monthly load, and run the same table again.

In [ ]:
bl_raw = pd.read_pickle('BankloanCleanNewVars.pkl')
bl_raw_train, bl_raw_test = train_test_split(bl_raw.drop(columns='customer'), test_size=0.3,
                                             random_state=20251023, stratify=bl_raw['Default'])

# A drifted current sample: weight towards young, high monthly-load applicants
w = np.exp(-0.06 * (bl_raw_test['Age'] - 20) + 2.0 * bl_raw_test['MonthlyLoad'])
# The WoE test set (Lab 5) and this raw split use the same seed, so rows are aligned. Check it.
assert np.array_equal(bl_test['Default'].values, bl_raw_test['Default'].values), 'row order differs between the WoE and raw test sets'
pos = rng.choice(len(bl_raw_test), size=len(bl_raw_test), replace=True, p=(w / w.sum()).values)
bl_test_drift = bl_test.iloc[pos].reset_index(drop=True)

rows = []
for c in woe_cols:
    labels = sorted(set(bl_train[c].round(6)) | set(bl_test_drift[c].round(6)))
    T = shares(bl_train[c].round(6), labels=labels)
    A = shares(bl_test_drift[c].round(6), labels=labels)
    t = stats.ttest_ind(bl_test_drift[c], bl_train[c], equal_var=False)
    rows.append({'attribute': c, 'SSI': ssi(A, T), 't p-value': t.pvalue, 'light': ssi_light(ssi(A, T))})
paint(pd.DataFrame(rows).set_index('attribute'))

### Annex B: adversarial validation

Instead of testing one attribute at a time, train a classifier to tell the development sample from the current sample. If it cannot (AUC near 0.5), there is no shift the model could be reacting to. If it can, its feature importance tells you **which** attributes drifted. We run it on the random split (should fail) and on the drifted sample (should succeed).

In [ ]:
def adversarial_auc(dev_X, cur_X, seed=0):
    X = pd.concat([dev_X, cur_X], ignore_index=True)
    y = np.r_[np.zeros(len(dev_X)), np.ones(len(cur_X))]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=seed)
    clf.fit(Xtr, ytr)
    auc = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
    imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    return auc, imp

auc_rand, imp_rand = adversarial_auc(bl_train[woe_cols], bl_test[woe_cols])
auc_drft, imp_drft = adversarial_auc(bl_train[woe_cols], bl_test_drift[woe_cols])
print(f'Adversarial AUC, random split : {auc_rand:.3f}')
print(f'Adversarial AUC, drifted set  : {auc_drft:.3f}')
pd.DataFrame({'importance (random split)': imp_rand, 'importance (drifted)': imp_drft}).style.format(precision=3)

## 2. Level 1: discrimination

Level 1 asks whether the model still **ranks** risk. The metric is the AUC. From the slides:

1. the AR bands (0.4 to 0.6 acceptable, 0.6 to 0.8 excellent) and the **AR difference with the development model** (< 5 points green, 5 to 10 yellow, > 10 red);
2. the ECB's **test of the AUC drop**: $S = (\text{AUC}_{\text{dev}} - \text{AUC}_{t}) / \hat{s}_t$, with the standard error of the current AUC from DeLong et al. (1988), and $p = 1 - \Phi(S)$.

The DeLong variance is a U-statistic computation; the function below is the standard fast implementation.

In [ ]:
# Define the DeLong AUC and variance functions, and the DeLong paired test for two AUCs on the same sample
def delong_auc(y, s):
    """AUC and its DeLong variance (Sun & Xu 2014 fast algorithm)."""
    y = np.asarray(y).astype(int); s = np.asarray(s, float)
    pos, neg = s[y == 1], s[y == 0]
    m, n = len(pos), len(neg)
    allv = np.r_[pos, neg]
    r_all = stats.rankdata(allv)              # midranks
    r_pos = stats.rankdata(pos); r_neg = stats.rankdata(neg)
    auc = (r_all[:m].sum() - m * (m + 1) / 2) / (m * n)
    v10 = (r_all[:m] - r_pos) / n             # structural components, positives
    v01 = 1 - (r_all[m:] - r_neg) / m         # structural components, negatives
    var = np.var(v10, ddof=1) / m + np.var(v01, ddof=1) / n
    return auc, var

def delong_paired_test(y, s1, s2):
    """Two-sided DeLong test for AUC(s1) = AUC(s2) on the same sample."""
    y = np.asarray(y).astype(int)
    aucs, comps = [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
        r_all = stats.rankdata(np.r_[pos, neg])
        v10 = (r_all[:m] - stats.rankdata(pos)) / n
        v01 = 1 - (r_all[m:] - stats.rankdata(neg)) / m
        aucs.append(v10.mean()); comps.append((v10, v01))
    m, n = len(comps[0][0]), len(comps[0][1])
    S10 = np.cov(comps[0][0], comps[1][0]); S01 = np.cov(comps[0][1], comps[1][1])
    S = S10 / m + S01 / n
    diff = aucs[0] - aucs[1]
    z = diff / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return aucs, z, 2 * stats.norm.sf(abs(z))

# Calculate the development AUC and AR, and the monitoring AUCs and ARs by year
auc_dev, var_dev = delong_auc(dev['Default'], dev['Probs'])
ar_dev = 2 * auc_dev - 1

def ar_light(diff_pts):
    return 'green' if diff_pts < 5 else ('yellow' if diff_pts < 10 else 'red')

rows = []
for y in MON_YEARS:
    cur = mon[mon['Year'] == y]
    auc_y, var_y = delong_auc(cur['Default'], cur['Probs'])
    se = np.sqrt(var_y)
    S = (auc_dev - auc_y) / se
    rows.append({'Year': y, 'n': len(cur), 'defaults': int(cur['Default'].sum()),
                 'AUC': auc_y, 'AUC 95% CI low': auc_y - 1.96 * se, 'AUC 95% CI high': auc_y + 1.96 * se,
                 'AR': 2 * auc_y - 1, 'AR drop (points)': 100 * (ar_dev - (2 * auc_y - 1)),
                 'S': S, 'p (AUC drop)': stats.norm.sf(S)})
lvl1 = pd.DataFrame(rows).set_index('Year')
lvl1['light'] = lvl1['AR drop (points)'].map(ar_light)
print(f'Development AUC = {auc_dev:.3f} (AR = {ar_dev:.3f}), n = {len(dev)}, defaults = {int(dev["Default"].sum())}')
paint(lvl1)

Read the two columns together. The AR-difference traffic light is a convention; the p-value tells you whether the drop is more than sampling noise given this year's sample size. With 60 defaults a year the confidence intervals are wide, and a 5-point drop in AR can be well inside them. A supervisor will accept "the drop is not significant" only if you also show the interval.

### Annex C: Level 1 for a machine learning model, the SHAP sign check

For a logistic scorecard, the Level 1 review checks that coefficient signs make economic sense. An XGBoost model has no coefficients, so the lecture replaces that row with an **explainability review**: are the SHAP contributions of each variable in the direction we expect? We retrain the Lab 7 XGBoost on bankloan (fixed hyper-parameters, no grid search) and correlate each variable with its SHAP values.

In [ ]:
import shap

# SHAP values for the XGBoost model on the bankloan data, to check whether the signs of the SHAP values match the expected signs of the variables.
expected_sign = {'Age': -1, 'Employ': -1, 'Address': -1, 'Income': -1,
                 'Leverage': +1, 'Creddebt': +1, 'OthDebt': +1, 'MonthlyLoad': +1, 'OthDebtRatio': +1}

# Train an XGBoost model on the raw bankloan data, and compute SHAP values on the test set.
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['Education'])],
                        remainder='passthrough')
xgb_bl = XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=200, objective='binary:logistic',
                       scale_pos_weight=(bl_raw_train['Default'] == 0).sum() / (bl_raw_train['Default'] == 1).sum(),
                       random_state=20260909, n_jobs=2, verbosity=0)
xgb_pipe = Pipeline([('pre', pre), ('clf', xgb_bl)])
xgb_pipe.fit(bl_raw_train.drop(columns='Default'), bl_raw_train['Default'])

X_test_t = pd.DataFrame(pre.transform(bl_raw_test.drop(columns='Default')), columns=pre.get_feature_names_out())
explainer = shap.TreeExplainer(xgb_bl, feature_names=X_test_t.columns)
shap_vals = explainer.shap_values(X_test_t, check_additivity=False)

# Check whether the signs of the SHAP values match the expected signs of the variables.
rows = []
for j, c in enumerate(X_test_t.columns):
    base = c.replace('remainder__', '')
    if base not in expected_sign:
        continue
    rho_s = stats.spearmanr(X_test_t[c], shap_vals[:, j]).statistic
    rows.append({'variable': base, 'expected sign': expected_sign[base],
                 'Spearman(variable, SHAP)': rho_s,
                 'mean |SHAP|': np.abs(shap_vals[:, j]).mean(),
                 'light': 'green' if np.sign(rho_s) == expected_sign[base] else 'red'})
paint(pd.DataFrame(rows).set_index('variable').sort_values('mean |SHAP|', ascending=False))

A red row here is the ML equivalent of a wrong coefficient sign: either the economics is wrong, or the variable is a proxy for something else (Lecture 11 will call this a fairness question). Here the red row is the other-debt ratio, a variable whose expected sign is debatable once other-debt and income are already in the model. The `shap.dependence_plot` from Lab 7 is the tool to see which of the two it is.

## 3. Level 2: calibration

Level 2 asks whether the promised PD matches the realised default rate. All tests below run per grade and per monitoring year, against the development long-run PD (the TTC PD). We follow the order of the slides.

### Binomial test

$H_0$: the grade PD is correct, $H_A$: it is underestimated. With $n$ obligors and $d$ defaults, the exact p-value is $P(X \ge d)$ under $\text{Bin}(n, PD)$. The normal approximation gives the critical default rate $p^* = PD + \Phi^{-1}(\alpha)\sqrt{PD(1-PD)/n}$. The lecture's convention: $\alpha$ is a **confidence level**; three lights at 95% and 99%.

In [ ]:
def light_95_99(p):
    """Three-colour light for a p-value: not significant at 95% / at 95% not 99% / at 99%."""
    if p is None or np.isnan(p):
        return 'grey'
    return 'green' if p > 0.05 else ('yellow' if p > 0.01 else 'red')

def binomial_test(n, d, pd_):
    if n == 0:
        return np.nan, np.nan, np.nan
    p_exact = stats.binom.sf(d - 1, n, pd_) if d > 0 else 1.0
    z = (d - n * pd_) / np.sqrt(n * pd_ * (1 - pd_))
    p_norm = stats.norm.sf(z)
    p_star99 = pd_ + stats.norm.ppf(0.99) * np.sqrt(pd_ * (1 - pd_) / n)
    return p_exact, p_norm, p_star99

def grade_year_table(year):
    cur = mon[mon['Year'] == year]
    t = cur.groupby('Grade').agg(n=('Default', 'size'), d=('Default', 'sum')).reindex(GRADES).fillna(0).astype(int)
    t['DR'] = t['d'] / t['n'].replace(0, np.nan)
    t['PD_TTC'] = master['PD_TTC']
    return t

def binomial_by_grade(year):
    t = grade_year_table(year)
    out = t.apply(lambda r: pd.Series(binomial_test(r['n'], r['d'], r['PD_TTC']),
                                      index=['p exact', 'p normal', 'p* (99%)']), axis=1)
    t = pd.concat([t, out], axis=1)
    t['light'] = t['p exact'].map(light_95_99)
    return t

paint(binomial_by_grade(MON_YEARS[-1]))

### Jeffreys test

The ECB's preferred calibration test. With the Jeffreys prior $\text{Beta}(\tfrac12, \tfrac12)$, the posterior for the grade PD is $\text{Beta}(d + \tfrac12,\; n - d + \tfrac12)$. The **p-value** is the posterior CDF evaluated at the estimated PD: a small value says the PD is underestimated; a value near 1 says the PD is conservative. For IFRS 9 you also want the **two-sided** credible interval, because over-estimation matters there too.

In [ ]:
def jeffreys(n, d, pd_, alpha=0.95):
    if n == 0:
        return np.nan, np.nan, np.nan
    a, b = d + 0.5, n - d + 0.5
    p_value = stats.beta.cdf(pd_, a, b)                    # one-sided: P(true PD <= estimate | data)
    lo, hi = stats.beta.ppf([(1 - alpha) / 2, (1 + alpha) / 2], a, b)
    return p_value, lo, hi

def jeffreys_by_grade(year, alpha=0.95):
    t = grade_year_table(year)
    out = t.apply(lambda r: pd.Series(jeffreys(r['n'], r['d'], r['PD_TTC'], alpha),
                                      index=['Jeffreys p', 'CI low', 'CI high']), axis=1)
    t = pd.concat([t, out], axis=1)
    t['light'] = t['Jeffreys p'].map(light_95_99)
    t['PD inside 2-sided CI'] = (t['PD_TTC'] >= t['CI low']) & (t['PD_TTC'] <= t['CI high'])
    return t

paint(jeffreys_by_grade(MON_YEARS[-1]))

Compare the Jeffreys p-values with the exact binomial p-values above. They move together, but Jeffreys behaves when $d = 0$ (the binomial p-value is then 1 by construction) and its interval has good frequentist coverage in small samples (Pull & Hurlin, 2025). That is why the ECB uses it for low-default portfolios.

### Hosmer–Lemeshow test

One statistic for all grades at once:

$$T = \sum_{k=1}^{K} \frac{(n_k\,PD_k - d_k)^2}{n_k\,PD_k\,(1 - PD_k)} \sim \chi^2_K$$

It assumes independent defaults, like the binomial test, and it is two-sided: over- and under-estimation both add to $T$.

In [ ]:
def hosmer_lemeshow(year):
    t = grade_year_table(year)
    t = t[t['n'] > 0]
    T = float(np.sum((t['n'] * t['PD_TTC'] - t['d']) ** 2 / (t['n'] * t['PD_TTC'] * (1 - t['PD_TTC']))))
    return T, len(t), stats.chi2.sf(T, len(t))

hl = pd.DataFrame([dict(zip(['T', 'df', 'p-value'], hosmer_lemeshow(y)), Year=y) for y in MON_YEARS]).set_index('Year')
hl['light'] = hl['p-value'].map(light_95_99)
paint(hl)

### Normal test (multi-period)

The binomial and Jeffreys tests look at one year. The normal test in the lecture pools the years $t = 1, \dots, T$ for a **single grade**: $H_0$ says no year's true PD exceeds its forecast. With $DR_t$ the realised rates,

$$\frac{\sum_t (DR_t - PD_t)}{\sqrt{T}\,\tau} > z_\alpha, \qquad \tau^2 = \frac{1}{T-1}\left[\sum_t (DR_t - PD_t)^2 - \frac{1}{T}\Big(\sum_t (DR_t - PD_t)\Big)^2\right]$$

It is the natural test for a **TTC** PD: a through-the-cycle PD is not supposed to match any single year, only the average over the cycle (the "risk rating philosophy" slide).

In [ ]:
def normal_test(grade, years):
    dr = np.array([grade_year_table(y).loc[grade, 'DR'] for y in years], float)
    dr = np.nan_to_num(dr)
    e = dr - master.loc[grade, 'PD_TTC']
    T = len(e)
    tau2 = (np.sum(e ** 2) - np.sum(e) ** 2 / T) / (T - 1)
    z = np.sum(e) / (np.sqrt(T) * np.sqrt(tau2)) if tau2 > 0 else np.nan
    return z, stats.norm.sf(z)

rows = []
for g in GRADES:
    z, p = normal_test(g, MON_YEARS)
    rows.append({'Grade': g, 'z': z, 'p-value': p, 'light': light_95_99(p)})
paint(pd.DataFrame(rows).set_index('Grade'))

The p-values of 1.000 for G3 and G5 are not a pass, they are a message: realised default rates sit far below the long-run PD in every year. The one-sided test does not flag conservatism, which is why the lecture suggests a "dark green" light for PDs that are too prudent.

### The PIT / TTC point, in numbers

Count how many grade-years the one-year binomial test flags, and compare with the pooled test above. A TTC PD tested year by year will fail in bad years by construction. The multi-period view is the one that matches what the PD promised.

In [ ]:
# Summarise the one-year binomial tests by grade and year, and the pooled normal test by grade.
flags = pd.DataFrame({y: binomial_by_grade(y)['light'] for y in MON_YEARS})
print('Grade-years flagged by the one-year binomial test (yellow or red):',
      int((flags != 'green').sum().sum()), 'of', flags.size)
print('Grades flagged by the pooled normal test:',
      int(sum(light_95_99(normal_test(g, MON_YEARS)[1]) != 'green' for g in GRADES)), 'of', len(GRADES))
flags.style.apply(lambda col: [f'background-color: {LIGHTS[v]}' for v in col])

### Correlated defaults: the one-factor (Vasicek) test

Every test so far assumed independent defaults. Under the one-factor model of the calibration lab, with asset correlation $\rho$, the critical default rate of a large portfolio at confidence level $\alpha$ is

$$DR^* = \Phi\!\left[\frac{\Phi^{-1}(PD) + \sqrt{\rho}\,\Phi^{-1}(\alpha)}{\sqrt{1-\rho}}\right]$$

For a **finite** portfolio the default count is binomial given the state of the economy, so we simulate: draw $Z$, compute the conditional PD, draw defaults. Compare the two critical values with the binomial $p^*$: correlation moves the bar a long way.

In [ ]:
def vasicek_dr_star(pd_, rho, alpha=0.99):
    return stats.norm.cdf((stats.norm.ppf(pd_) + np.sqrt(rho) * stats.norm.ppf(alpha)) / np.sqrt(1 - rho))

def vasicek_finite_n(pd_, rho, n, alpha=0.99, sims=200_000, seed=1):
    r = np.random.default_rng(seed)
    z = r.standard_normal(sims)
    p_cond = stats.norm.cdf((stats.norm.ppf(pd_) - np.sqrt(rho) * z) / np.sqrt(1 - rho))
    d = r.binomial(n, p_cond)
    return np.quantile(d / n, alpha)

RHO = 0.15   # Basel retail residential mortgage; the calibration lab used the same value
year = MON_YEARS[-1]
t = grade_year_table(year)
t['p* binomial (99%)'] = t.apply(lambda r: binomial_test(r['n'], r['d'], r['PD_TTC'])[2], axis=1)
t['DR* Vasicek, n -> inf'] = t['PD_TTC'].map(lambda p: vasicek_dr_star(p, RHO))
t['DR* Vasicek, finite n'] = t.apply(lambda r: vasicek_finite_n(r['PD_TTC'], RHO, int(r['n'])), axis=1)
t['light (Vasicek finite n)'] = np.where(t['DR'] > t['DR* Vasicek, finite n'], 'red',
                                 np.where(t['DR'] > t.apply(lambda r: vasicek_finite_n(r['PD_TTC'], RHO, int(r['n']), 0.95), axis=1), 'yellow', 'green'))
paint(t, col='light (Vasicek finite n)')

The finite-$n$ critical value sits between the binomial and the asymptotic Vasicek value: small grades add sampling noise on top of the systematic factor. The binomial test is the most severe of the three. That is the sense in which the lecture calls it a *prudential* test: fine as an early-warning signal, too strict as a decision rule when defaults are correlated. The 2026 EBA staff paper generalises this correction to serial correlation as well.

**Exercise.** *Set `RHO = 0.04` (QRRE) and `RHO = 0.24` (corporate upper bound) and rerun the cell. How many grades change colour?*

### Brier score, its decomposition, and the Spiegelhalter test

The Brier score $BS = \frac{1}{n}\sum_i (\widehat{PD}_i - \theta_i)^2$ measures overall forecast accuracy. Murphy (1973) splits it over the grades into

$$BS = \underbrace{\sum_k \tfrac{n_k}{n}\,(\widehat{PD}_k - DR_k)^2}_{\text{reliability (Level 2)}} \;-\; \underbrace{\sum_k \tfrac{n_k}{n}\,(DR_k - DR)^2}_{\text{resolution (Level 1)}} \;+\; \underbrace{DR\,(1-DR)}_{\text{uncertainty}}$$

so a single number carries both levels. Spiegelhalter (1986) turns calibration into a z-test with no binning at all:

$$Z = \frac{\sum_i (\theta_i - \widehat{PD}_i)(1 - 2\widehat{PD}_i)}{\sqrt{\sum_i (1 - 2\widehat{PD}_i)^2\,\widehat{PD}_i\,(1 - \widehat{PD}_i)}} \sim N(0, 1)$$

We evaluate both with the grade PD as the forecast (that is the number the bank reports), and draw the reliability diagram with the expected calibration error (ECE).

In [ ]:
def brier_block(year):
    cur = mon[mon['Year'] == year].copy()
    cur['PD_hat'] = cur['Grade'].map(master['PD_TTC'])
    theta, p = cur['Default'].values, cur['PD_hat'].values
    bs = np.mean((p - theta) ** 2)
    g = cur.groupby('Grade').agg(n=('Default', 'size'), DR=('Default', 'mean'), PD=('PD_hat', 'first'))
    w = g['n'] / g['n'].sum(); DR = theta.mean()
    reliability = float(np.sum(w * (g['PD'] - g['DR']) ** 2))
    resolution  = float(np.sum(w * (g['DR'] - DR) ** 2))
    uncertainty = DR * (1 - DR)
    Z = np.sum((theta - p) * (1 - 2 * p)) / np.sqrt(np.sum((1 - 2 * p) ** 2 * p * (1 - p)))
    ece = float(np.sum(w * np.abs(g['PD'] - g['DR'])))
    return dict(Brier=bs, reliability=reliability, resolution=resolution, uncertainty=uncertainty,
                check=reliability - resolution + uncertainty, Spiegelhalter_Z=Z,
                p_two_sided=2 * stats.norm.sf(abs(Z)), ECE=ece), g

bt = pd.DataFrame({y: brier_block(y)[0] for y in MON_YEARS}).T
bt.index.name = 'Year'
bt['light'] = bt['p_two_sided'].map(light_95_99)
paint(bt)

In [ ]:
_, g = brier_block(MON_YEARS[-1])
fig, ax = plt.subplots(figsize=(5.5, 5.5))
lim = max(g['PD'].max(), g['DR'].max()) * 1.1
ax.plot([0, lim], [0, lim], '--', color='grey', label='perfect calibration')
ax.plot(g['PD'], g['DR'], 'o-', color='#4B2E83', label=f'observed DR per grade, {MON_YEARS[-1]}')
for k, r in g.iterrows():
    ax.vlines(r['PD'], min(r['PD'], r['DR']), max(r['PD'], r['DR']), color='#c0392b', lw=1)
    ax.annotate(k, (r['PD'], r['DR']), textcoords='offset points', xytext=(5, -10), fontsize=8)
ax.set_xlabel('Long-run PD of the grade'); ax.set_ylabel('Observed default rate')
ax.set_title('Reliability diagram'); ax.legend(loc='upper left')
plt.show()

The `check` column reproduces the Brier score from its three parts, so the decomposition is exact. Reliability is what Level 2 tests; resolution is what Level 1 tests. When a model is recalibrated, reliability falls and resolution stays put. When it is redeveloped, resolution moves.

Look at the Spiegelhalter column against the binomial and Jeffreys tables above. Every one-sided test is green, and the Spiegelhalter test is red in the first monitoring year. The PDs were too **high** that year (a negative $Z$): the one-sided Basel tests do not see over-estimation, a two-sided test does. Under IFRS 9 that is a finding.

## 4. The traffic-light dashboard and the action scheme

Now we put the pieces together into the dashboard of the lecture (one table per level, three colours) and run it for every monitoring year. The rules are the ones on the slides:

| Level | Test | Green | Yellow | Red |
|---|---|---|---|---|
| 2 | Binomial (pooled), Hosmer–Lemeshow, Vasicek, Normal, Spiegelhalter | not significant at 95% | at 95%, not 99% | at 99% |
| 1 | AR difference with development | < 5 points | 5 to 10 | > 10 |
| 1 | AUC drop test | p > 0.05 | 0.01 to 0.05 | < 0.01 |
| 0 | SSI (grades), SSI (score) | < 0.10 | 0.10 to 0.25 | > 0.25 |
| 0 | Herfindahl vs development | inside bootstrap CI | within 0.10 | otherwise |

The pooled binomial and Vasicek tests use the whole portfolio of the year against its exposure-weighted PD, which is what you would report to a committee before drilling into the grades.

In [ ]:
def dashboard(year):
    cur = mon[mon['Year'] == year]
    t = grade_year_table(year)
    n, d = int(t['n'].sum()), int(t['d'].sum())
    pd_port = float(np.sum(t['n'] * t['PD_TTC']) / n)

    p_bin = binomial_test(n, d, pd_port)[0]
    p_jef = jeffreys(n, d, pd_port)[0]
    T, df, p_hl = hosmer_lemeshow(year)
    dr = d / n
    dr95 = vasicek_finite_n(pd_port, RHO, n, 0.95); dr99 = vasicek_finite_n(pd_port, RHO, n, 0.99)
    vas_light = 'red' if dr > dr99 else ('yellow' if dr > dr95 else 'green')
    years_so_far = [y for y in MON_YEARS if y <= year]
    if len(years_so_far) >= 2:
        drs = np.array([mon[mon['Year'] == y]['Default'].mean() for y in years_so_far]); e = drs - pd_port
        tau2 = (np.sum(e ** 2) - np.sum(e) ** 2 / len(e)) / (len(e) - 1)
        p_norm = stats.norm.sf(np.sum(e) / (np.sqrt(len(e)) * np.sqrt(tau2))) if tau2 > 0 else np.nan
    else:
        p_norm = np.nan
    bb, _ = brier_block(year)

    auc_y, var_y = delong_auc(cur['Default'], cur['Probs'])
    ar_drop = 100 * (ar_dev - (2 * auc_y - 1))
    p_auc = stats.norm.sf((auc_dev - auc_y) / np.sqrt(var_y))

    s_grade = ssi(shares(cur['Grade'], labels=GRADES), T_grade)
    s_score = ssi(shares(cur['Probs'], dec_bins), T_score)
    h = herfindahl(shares(cur['Grade'], labels=GRADES))

    rows = [
        ('Level 2: calibration', 'Binomial, portfolio (p)',          p_bin, light_95_99(p_bin)),
        ('Level 2: calibration', 'Jeffreys, portfolio (p)',          p_jef, light_95_99(p_jef)),
        ('Level 2: calibration', 'Hosmer-Lemeshow across grades (p)', p_hl, light_95_99(p_hl)),
        ('Level 2: calibration', f'Vasicek finite n, rho={RHO} (DR)', dr,   vas_light),
        ('Level 2: calibration', 'Normal multi-period, portfolio (p)', p_norm, light_95_99(p_norm) if not np.isnan(p_norm) else 'grey'),
        ('Level 2: calibration', 'Spiegelhalter Z (two-sided p)',    bb['p_two_sided'], light_95_99(bb['p_two_sided'])),
        ('Level 1: discrimination', 'AR drop vs development (points)', ar_drop, ar_light(ar_drop)),
        ('Level 1: discrimination', 'AUC drop test (p)',              p_auc, light_95_99(p_auc)),
        ('Level 0: stability', 'SSI grades',                          s_grade, ssi_light(s_grade)),
        ('Level 0: stability', 'SSI score deciles',                   s_score, ssi_light(s_score)),
        ('Level 0: stability', 'Herfindahl (dev = %.3f)' % HI_dev,    h, 'green' if lo <= h <= hi else ('yellow' if abs(h - HI_dev) < 0.10 else 'red')),
    ]
    out = pd.DataFrame(rows, columns=['Level', 'Test', 'value', 'light']).set_index(['Level', 'Test'])
    return out

paint(dashboard(MON_YEARS[-1]))

In [ ]:
history = pd.concat({y: dashboard(y)['light'] for y in MON_YEARS}, axis=1)
history.style.apply(lambda col: [f'background-color: {LIGHTS[v]}' for v in col])

Read the history by rows. Level 2 is green throughout except for the two-sided Spiegelhalter test, which flags over-estimation in 2003 and, more weakly, in 2006. Level 1 never moves. Level 0 shows the persistent shift towards safer grades that the Herfindahl row picks up and the SSI does not, because the SSI thresholds are loose.

### The action scheme

The lecture's action scheme is a decision tree: calibration OK: keep the model; calibration not OK but discrimination OK: recalibrate; discrimination not OK but data stable: re-estimate the model; data not stable: re-develop on the new population (and only then re-test). We encode it and apply it to each year. "Not OK" means at least one red, or two yellows, within a level.

In [ ]:
def level_status(lights):
    lights = list(lights)
    return 'NOT OK' if (lights.count('red') >= 1 or lights.count('yellow') >= 2) else 'OK'

def action(year):
    d = dashboard(year)['light']
    cal = level_status(d.loc['Level 2: calibration'])
    dis = level_status(d.loc['Level 1: discrimination'])
    sta = level_status(d.loc['Level 0: stability'])
    if cal == 'OK':
        act = 'Continue using the model'
    elif dis == 'OK':
        act = 'Re-calibrate the grade PDs'
    elif sta == 'OK':
        act = 'Re-estimate the model on the same population'
    else:
        act = 'Population has shifted: re-develop the model, then re-test'
    return {'calibration': cal, 'discrimination': dis, 'data stability': sta, 'action': act}

pd.DataFrame({y: action(y) for y in MON_YEARS}).T

## 5. Backtesting LGD models

We now validate the two LGD models from the LGD lab: the elastic net and the XGBoost regressor, refit here on the same train/test split (same seed). The holdout set plays the role of the out-of-time sample. The lecture gives three groups of tests.

**Calibration.** Error $e_i = LGD_i^{obs} - LGD_i^{pred}$. The paired **t test** ($H_0: \mu_e = 0$, $H_A: \mu_e > 0$, loss under-predicted) and its non-parametric twin, the **Wilcoxon signed-rank test** on the median error.

**Discrimination and dispersion.** The theoretical distribution of $R^2$, MSE or MAD on a new sample is unknown, so Loterman et al. (2014) **bootstrap** the difference between train and test performance. Error **dispersion** is tested with the **F test** on the error variances (test vs train) and the **Ansari–Bradley** test as the non-parametric variant. For ranking, the ECB uses the **generalised AUC** (gAUC, Somers' D between predicted and realised LGD).

**Exposure-weighted measures.** The **loss shortfall** and the exposure-weighted **MAD** of Maarse (2012), with their traffic lights. The LGD file has no exposures, so we simulate EADs; in practice they come with the data.

In [ ]:
LGD_data = pd.read_csv('LGD.csv')
X_lgd, y_lgd = LGD_data.drop(columns='LGD'), LGD_data['LGD']
x_train, x_test, y_train, y_test = train_test_split(X_lgd, y_lgd, test_size=0.33, random_state=20201209)

lgd_enet = ElasticNetCV(l1_ratio=np.arange(0.01, 1.01, 0.05), alphas=np.logspace(-4, 0, 10), cv=3, random_state=20201209, n_jobs=2).fit(x_train, y_train)
lgd_xgb  = XGBRegressor(max_depth=3, learning_rate=0.05, n_estimators=300, subsample=0.8, colsample_bytree=0.8,
                        random_state=20201209, n_jobs=2, verbosity=0).fit(x_train, y_train)

models = {'Elastic net': lgd_enet, 'XGBoost': lgd_xgb}
preds = {k: {'train': np.clip(m.predict(x_train), 0, 1), 'test': np.clip(m.predict(x_test), 0, 1)} for k, m in models.items()}

pd.DataFrame({k: {'RMSE train': np.sqrt(mean_squared_error(y_train, v['train'])),
                  'RMSE test':  np.sqrt(mean_squared_error(y_test, v['test'])),
                  'R2 train': r2_score(y_train, v['train']), 'R2 test': r2_score(y_test, v['test'])}
              for k, v in preds.items()}).T.style.format(precision=4)

### Calibration: t test and Wilcoxon

In [ ]:
rows = []
for k, v in preds.items():
    e = y_test.values - v['test']
    t = stats.ttest_1samp(e, 0.0, alternative='greater')
    w = stats.wilcoxon(e[e != 0], alternative='greater')
    rows.append({'model': k, 'mean error': e.mean(), 'median error': np.median(e),
                 't statistic': t.statistic, 't p (one-sided)': t.pvalue,
                 'Wilcoxon p (one-sided)': w.pvalue, 'light': light_95_99(min(t.pvalue, w.pvalue))})
paint(pd.DataFrame(rows).set_index('model'))

A positive mean error means realised losses exceed predictions: the model is optimistic, which is the direction a supervisor cares about. The one-sided tests answer exactly that question. For IFRS 9 you would run them two-sided.

### Discrimination and dispersion: bootstrap, F test, Ansari–Bradley, gAUC

In [ ]:
def loterman_bootstrap(model, x_tr, y_tr, x_te, y_te, metric, B=1000, seed=0):
    """H0: P_test = P_train. Pool, resample train/test of the original sizes, recompute the gap."""
    r = np.random.default_rng(seed)
    X = pd.concat([x_tr, x_te]); y = np.r_[y_tr, y_te]
    pred = np.clip(model.predict(X), 0, 1)
    n_tr = len(x_tr)
    observed = metric(y_te, np.clip(model.predict(x_te), 0, 1)) - metric(y_tr, np.clip(model.predict(x_tr), 0, 1))
    gaps = np.empty(B)
    for b in range(B):
        idx = r.permutation(len(y))
        i_tr, i_te = idx[:n_tr], idx[n_tr:]
        gaps[b] = metric(y[i_te], pred[i_te]) - metric(y[i_tr], pred[i_tr])
    return observed, gaps

def gauc(y_true, y_pred):
    """Generalised AUC = (Somers' D + 1) / 2, ranking realised LGD by predicted LGD."""
    return (stats.somersd(y_pred, y_true).statistic + 1) / 2

rows = []
for k, v in preds.items():
    e_tr, e_te = y_train.values - v['train'], y_test.values - v['test']
    obs_mse, gaps_mse = loterman_bootstrap(models[k], x_train, y_train, x_test, y_test, mean_squared_error)
    obs_r2,  gaps_r2  = loterman_bootstrap(models[k], x_train, y_train, x_test, y_test, r2_score)
    F = np.var(e_te, ddof=1) / np.var(e_tr, ddof=1)
    p_F = stats.f.sf(F, len(e_te) - 1, len(e_tr) - 1)
    ab = stats.ansari(e_te, e_tr)
    rows.append({'model': k,
                 'MSE gap test-train': obs_mse, 'bootstrap p (MSE worse)': np.mean(gaps_mse >= obs_mse),
                 'R2 gap test-train': obs_r2,   'bootstrap p (R2 worse)':  np.mean(gaps_r2 <= obs_r2),
                 'F (var test / var train)': F, 'F p (one-sided)': p_F, 'Ansari-Bradley p': ab.pvalue,
                 'gAUC train': gauc(y_train, v['train']), 'gAUC test': gauc(y_test, v['test'])})
lgd_disc = pd.DataFrame(rows).set_index('model')
lgd_disc['light'] = lgd_disc[['bootstrap p (MSE worse)', 'F p (one-sided)']].min(axis=1).map(light_95_99)
paint(lgd_disc)

The bootstrap p-value is the share of resampled train/test splits with a gap at least as bad as the one observed. A small value says the holdout performance is worse than what resampling alone would produce: the model does not generalise. The XGBoost model will typically show the larger gap: it fits the training set more closely, so the out-of-sample drop is bigger. The gAUC on the test set tells you whether the ranking survives even if the level does not.

### Exposure-weighted measures: loss shortfall and MAD

$$LS = 1 - \frac{\sum_i \widehat{LGD}_i\, EAD_i}{\sum_i LGD_i\, EAD_i}, \qquad MAD = \frac{\sum_i |LGD_i - \widehat{LGD}_i|\, EAD_i}{\sum_i EAD_i}$$

Maarse (2012) traffic lights: LS > 0 or LS ≤ −0.20 red, −0.20 < LS ≤ −0.10 yellow, −0.10 < LS ≤ 0 green; MAD ≤ 0.10 green, 0.10 to 0.20 yellow, above 0.20 red.

In [ ]:
ead_test = pd.Series(rng.lognormal(mean=10, sigma=0.8, size=len(y_test)), index=y_test.index)   # simulated exposures

def loss_shortfall(lgd_obs, lgd_pred, ead):
    return 1 - np.sum(lgd_pred * ead) / np.sum(lgd_obs * ead)

def mad_w(lgd_obs, lgd_pred, ead):
    return np.sum(np.abs(lgd_obs - lgd_pred) * ead) / np.sum(ead)

def ls_light(v):  return 'red' if (v > 0 or v <= -0.20) else ('yellow' if v <= -0.10 else 'green')
def mad_light(v): return 'green' if v <= 0.10 else ('yellow' if v <= 0.20 else 'red')

rows = []
for k, v in preds.items():
    LS = loss_shortfall(y_test.values, v['test'], ead_test.values); MAD = mad_w(y_test.values, v['test'], ead_test.values)
    rows.append({'model': k, 'Loss shortfall': LS, 'LS light': ls_light(LS), 'MAD (EAD-weighted)': MAD, 'MAD light': mad_light(MAD)})
ew = pd.DataFrame(rows).set_index('model')
ew.style.apply(lambda r: [f'background-color: {LIGHTS[r["LS light"]]}' if c == 'LS light' else
                          (f'background-color: {LIGHTS[r["MAD light"]]}' if c == 'MAD light' else '') for c in r.index], axis=1).format(precision=4)

The loss shortfall is a one-number, conservative check: a positive value means the model under-predicts the total loss on the book, whatever it does obligor by obligor. The MAD is the accuracy check. A model can pass the first and fail the second (large errors that cancel), which is why the lecture reports both.

## 6. Benchmarking

Benchmarking compares the model with an external or internal reference. We have an internal one: the XGBoost model from Annex C is a **challenger** to the Lab 6 scorecard (the **champion**). The lecture gives three rank statistics for how much two ratings agree: **Spearman's rank correlation**, **Kendall's tau** and **Goodman–Kruskal gamma**. We first reproduce the five-customer example of the slides as a check on our functions, then run them on the bankloan test set.

In [ ]:
def kendall_tau_a(x, y):
    """Kendall's tau-a: (concordant - discordant) / number of pairs. Ties count as neither."""
    x, y = np.asarray(x), np.asarray(y); n = len(x); A = B = 0
    for i in range(n):
        for j in range(i + 1, n):
            s = np.sign(x[i] - x[j]) * np.sign(y[i] - y[j])
            A += s > 0; B += s < 0
    return (A - B) / (n * (n - 1) / 2), A, B

def gk_gamma(x, y):
    _, A, B = kendall_tau_a(x, y)
    return (A - B) / (A + B)

# Slide example: five customers, internal score vs FICO
internal = [20, 35, 15, 25, 20]; fico = [680, 580, 640, 720, 700]
tau, A, B = kendall_tau_a(internal, fico)
d2 = (stats.rankdata(internal) - stats.rankdata(fico)) ** 2
print(f"Spearman, no-ties formula 1 - 6*sum(d^2)/(n(n^2-1)) = {1 - 6 * d2.sum() / (5 * 24):.3f}  (slide: -0.025)")
print(f"Spearman, exact (Pearson on ranks, handles the tie)  = {stats.spearmanr(internal, fico).statistic:.3f}")
print(f"Concordant = {A}, discordant = {B}, Kendall tau-a = {tau:.2f}  (slide: 0.1)")
print(f"Goodman-Kruskal gamma = {gk_gamma(internal, fico):.2f}  (slide: 0.11)")

In [ ]:
# Champion: the Lab 6 scorecard (logistic regression on WoE variables)
champion = LogisticRegression(penalty='elasticnet', C=0.359, l1_ratio=0.100, solver='saga', class_weight='balanced',
                              max_iter=1000, tol=1e-6, random_state=20190301).fit(bl_train[woe_cols], bl_train['Default'])
p_champ = champion.predict_proba(bl_test[woe_cols])[:, 1]
p_chall = xgb_pipe.predict_proba(bl_raw_test.drop(columns='Default'))[:, 1]
y_bl = bl_raw_test['Default'].values


aucs, z, p = delong_paired_test(y_bl, p_champ, p_chall)
print(f"AUC champion (scorecard) = {aucs[0]:.3f}, AUC challenger (XGBoost) = {aucs[1]:.3f}")
print(f"DeLong paired test of equal AUCs: z = {z:.2f}, p = {p:.3f}")
print(f"Spearman between the two PDs   : {stats.spearmanr(p_champ, p_chall).statistic:.3f}")
print(f"Kendall tau-b (scipy, handles ties): {stats.kendalltau(p_champ, p_chall).statistic:.3f}")
print(f"Goodman-Kruskal gamma            : {gk_gamma(np.round(p_champ, 3), np.round(p_chall, 3)):.3f}")

The two Spearman values differ because customers 1 and 5 are tied on the internal score: the slide uses the no-ties approximation, scipy computes the exact Pearson correlation of the ranks. With ties, use the exact one.

The agreement statistics answer "do the two models rank the same customers as risky?", the DeLong test answers "is one of them better?". A challenger that agrees strongly with the champion and is not significantly better does not justify its complexity (the ECB's rule for machine learning models). A challenger that disagrees strongly is a benchmarking finding in its own right: someone has to explain which customers they disagree on.

## 7. Closing

You now have a validation framework that reproduces every table of the lecture from data: stability, discrimination and calibration tests per grade and per year, the traffic-light dashboard, the action scheme, the LGD backtests and the benchmarking statistics. In practice the `dashboard()` function is what runs every month, and the rest of the notebook is what the validation team runs once a year.

**Exercises**

1. Change `RHO` to 0.04 and 0.24 and rerun the Vasicek cells. Which grades change colour, and why does the effect depend on the grade PD?
2. Aggregate the monitoring years in pairs and rerun the Jeffreys test. How much narrower is the credible interval? Which grades can now be tested at all?
3. Change the development window to the first six years and repeat the dashboard history. Does a longer development window make the monitoring years look better or worse? Why?
4. Write the one-page validation report for grade G3 in the last monitoring year: what was promised, what was observed, which tests were run, what colour, what action.

**Not covered here: IFRS 9.** The staging back-tests, lifetime PD term structures and scenario weights of the lecture need obligor-level histories over several years, which none of our datasets have. We will build a simulated panel for the stress-testing lab and validate the IFRS 9 components there.